# ❤️ CardioCheck Indonesia — Model Training
### Notebook Google Colab untuk Melatih Model Prediksi Risiko Penyakit Jantung

**Dataset:** heart_attack_prediction_indonesia.csv (158.355 data pasien)

> Jalankan semua cell secara berurutan. Di akhir, file `model.pkl` dan `encoders.json` akan terunduh otomatis.

## 📦 Install & Import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

print('✅ Semua library berhasil diimpor!')

## 📂 Upload Dataset
> Upload file `heart_attack_prediction_indonesia.csv` dari komputer Anda

In [ ]:
from google.colab import files
print('📤 Silakan upload file CSV dataset...')
uploaded = files.upload()

## 📊 Eksplorasi Data (EDA)

In [ ]:
# Load data
df = pd.read_csv('heart_attack_prediction_indonesia.csv')
print(f'📊 Shape dataset: {df.shape}')
print(f'📊 Jumlah baris: {df.shape[0]:,}')
print(f'📊 Jumlah kolom: {df.shape[1]}')
df.head()

In [ ]:
print('📋 Informasi Dataset:')
df.info()

In [ ]:
print('📊 Statistik Deskriptif:')
df.describe()

In [ ]:
# Distribusi target
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
counts = df['heart_attack'].value_counts()
axes[0].pie(counts, labels=['Tidak Serangan\nJantung', 'Serangan\nJantung'],
            autopct='%1.1f%%', colors=['#6bcb77', '#ff6b6b'],
            startangle=90, textprops={'fontsize': 12})
axes[0].set_title('Distribusi Target (heart_attack)', fontsize=14, fontweight='bold')

# Bar chart by age group
df['age_group'] = pd.cut(df['age'], bins=[25, 35, 45, 55, 65, 90],
                          labels=['25-35', '36-45', '46-55', '56-65', '66+'])
age_risk = df.groupby('age_group')['heart_attack'].mean() * 100
axes[1].bar(age_risk.index, age_risk.values, color='#ff6b6b', alpha=0.8, edgecolor='white')
axes[1].set_title('Risiko Serangan Jantung per Kelompok Usia', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Kelompok Usia')
axes[1].set_ylabel('Persentase Risiko (%)')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('eda_distribusi.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✅ Target distribution: {dict(counts)}')

In [ ]:
# Correlation heatmap (numerik saja)
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'age_group' in num_cols:
    num_cols.remove('age_group')

plt.figure(figsize=(16, 12))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn_r',
            center=0, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Korelasi Antar Fitur Numerik', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔧 Preprocessing

In [ ]:
# Drop kolom yang tidak dipakai
if 'age_group' in df.columns:
    df = df.drop('age_group', axis=1)

# Handle missing values
print('Missing values sebelum handling:')
print(df.isnull().sum()[df.isnull().sum() > 0])

df['alcohol_consumption'] = df['alcohol_consumption'].fillna('None')

print('\n✅ Missing values setelah handling:')
print(df.isnull().sum().sum(), 'total missing values')

In [ ]:
# Label Encoding untuk kolom kategorikal
cat_cols = [
    'gender', 'smoking_status', 'alcohol_consumption', 'physical_activity',
    'dietary_habits', 'stress_level', 'EKG_results', 'income_level',
    'region', 'air_pollution_exposure'
]

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = {cls: int(idx) for idx, cls in enumerate(le.classes_)}
    print(f'{col}: {encoders[col]}')

print('\n✅ Encoding selesai!')

In [ ]:
# Pilih fitur
feature_cols = [
    'age', 'gender', 'hypertension', 'diabetes', 'cholesterol_level',
    'obesity', 'waist_circumference', 'family_history', 'smoking_status',
    'alcohol_consumption', 'physical_activity', 'dietary_habits',
    'stress_level', 'sleep_hours', 'blood_pressure_systolic',
    'blood_pressure_diastolic', 'fasting_blood_sugar', 'cholesterol_hdl',
    'cholesterol_ldl', 'triglycerides', 'EKG_results',
    'previous_heart_disease', 'medication_usage'
]

X = df[feature_cols]
y = df['heart_attack']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'✅ Train size: {X_train.shape[0]:,}')
print(f'✅ Test size : {X_test.shape[0]:,}')
print(f'✅ Features  : {X_train.shape[1]}')

## 🤖 Training Model

In [ ]:
print('🔄 Training Random Forest...')
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)
print('✅ Training selesai!')

## 📈 Evaluasi Model

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print(f'🎯 Accuracy : {acc:.4f} ({acc*100:.2f}%)')
print(f'🎯 ROC-AUC  : {auc:.4f}')
print()
print('📊 Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Tidak Sakit Jantung', 'Sakit Jantung']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Tidak Sakit', 'Sakit Jantung'])
disp.plot(ax=axes[0], colorbar=False, cmap='RdYlGn')
axes[0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color='#ff6b6b', lw=2, label=f'ROC (AUC = {auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('evaluasi_model.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance
feat_imp = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 8))
colors = ['#ff6b6b' if imp > 0.065 else '#ffd93d' if imp > 0.05 else '#6bcb77'
          for imp in feat_imp['importance']]
plt.barh(feat_imp['feature'], feat_imp['importance'], color=colors, edgecolor='white', alpha=0.9)
plt.xlabel('Importance Score', fontsize=12)
plt.title('Feature Importance — Random Forest', fontsize=15, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n🏆 Top 10 Fitur Paling Penting:')
print(feat_imp.sort_values('importance', ascending=False).head(10).to_string(index=False))

## 💾 Simpan & Download Model

In [ ]:
# Simpan model
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Simpan encoders
with open('encoders.json', 'w') as f:
    json.dump(encoders, f, indent=2)

print('✅ model.pkl tersimpan!')
print('✅ encoders.json tersimpan!')
print()
print('📦 Mengunduh file...')

files.download('model.pkl')
files.download('encoders.json')
print('✅ Download selesai! Upload kedua file ini ke repository GitHub Anda.')

## 🎉 Selesai!

Upload `model.pkl` dan `encoders.json` ke repository GitHub bersama `app.py` dan `requirements.txt`, lalu deploy di [Streamlit Cloud](https://share.streamlit.io).